# Phase 3 — Test & Fix Feedback Loop

This notebook lets you inspect, trigger, and debug the Phase 3 pipeline.

**Sections:**
1. Project state — what files Phase 2 generated
2. Health check — verify test-runner is up
3. Run tests directly — quick pytest without the fix loop
4. Trigger Phase 3 — full loop (test → LLM fix → re-test)
5. View report — read `phase3_report.md`
6. Inject a bug — deliberately break a file to test the fix loop
7. Re-run Phase 3 — watch it detect and fix the bug

In [1]:
import os, subprocess, json, textwrap
from pathlib import Path
import requests

# ── paths ──────────────────────────────────────────────────────────────────
# Jupyter sees the shared volume at /home/jovyan/output
# test-runner sees the same volume at /data/output
PROJECT_DIR_LOCAL  = Path('/home/jovyan/output/project')   # for file ops in this notebook
PROJECT_DIR_RUNNER = '/data/output/project'                # passed to test-runner API
REPORT_PATH        = Path('/home/jovyan/output/phase3_report.md')

# ── service URLs (internal Docker network) ─────────────────────────────────
TEST_RUNNER_URL = 'http://test-runner:5001'
N8N_URL         = os.environ.get('N8N_BASE_URL', 'http://n8n:5678')
LITELLM_URL     = os.environ.get('LITELLM_BASE_URL', 'http://litellm:4000')
LITELLM_KEY     = os.environ.get('LITELLM_API_KEY', '')
N8N_WEBHOOK     = f'{N8N_URL}/webhook/agentic-phase3-001/webhook/phase3-run'

print('Config ready')
print(f'  Project  : {PROJECT_DIR_LOCAL}')
print(f'  Runner   : {TEST_RUNNER_URL}')
print(f'  Webhook  : {N8N_WEBHOOK}')

Config ready
  Project  : /home/jovyan/output/project
  Runner   : http://test-runner:5001
  Webhook  : http://n8n:5678/webhook/agentic-phase3-001/webhook/phase3-run


## 1. Project State

In [2]:
# List all project files (skip venv / caches)
SKIP = {'.venv', '__pycache__', '.pytest_cache', '.git'}

if not PROJECT_DIR_LOCAL.exists():
    print('⚠️  No project found — run Phase 1 + 2 first.')
else:
    files = sorted(
        f for f in PROJECT_DIR_LOCAL.rglob('*')
        if f.is_file() and not any(p in SKIP for p in f.parts)
        and f.suffix not in ('.pyc', '.pyo')
    )
    print(f'Files in project ({len(files)} total):\n')
    for f in files:
        rel = f.relative_to(PROJECT_DIR_LOCAL)
        size = f.stat().st_size
        print(f'  {str(rel):45s}  {size:>6} bytes')

Files in project (14 total):

  .gitignore                                        538 bytes
  Dockerfile                                        165 bytes
  MANIFEST.txt                                      108 bytes
  README.md                                        1542 bytes
  app.py                                            641 bytes
  csv_to_json_converter/main.py                     633 bytes
  csv_to_json_converter.py                          597 bytes
  open_meteo_weather.py                             858 bytes
  requirements.txt                                   47 bytes
  src/open_meteo_weather.py                        1080 bytes
  tests/conftest.py                                  95 bytes
  tests/test_open_meteo_weather.py                 1569 bytes
  tests/test_todo.py                                583 bytes
  todo.py                                           855 bytes


In [3]:
# View the contents of specific files
# Edit the list below to inspect different files
FILES_TO_SHOW = ['requirements.txt', 'tests/test_todo.py', 'tests/conftest.py']

for rel in FILES_TO_SHOW:
    path = PROJECT_DIR_LOCAL / rel
    print(f'\n{'─'*60}')
    print(f'  {rel}')
    print(f'{'─'*60}')
    if path.exists():
        print(path.read_text())
    else:
        print('  (file not found)')


────────────────────────────────────────────────────────────
  requirements.txt
────────────────────────────────────────────────────────────
click==8.1.3
requests==2.28.1
geocoder==1.38.1


────────────────────────────────────────────────────────────
  tests/test_todo.py
────────────────────────────────────────────────────────────
import unittest
from todo import Todo, TodoList

class TestTodo(unittest.TestCase):
    def test_todo_creation(self):
        todo = Todo(1, 'Buy milk')
        self.assertEqual(todo.title, 'Buy milk')

class TestTodoList(unittest.TestCase):
    def test_add_todo(self):
        todo_list = TodoList()
        todo = todo_list.add_todo('Buy eggs')
        self.assertEqual(todo.title, 'Buy eggs')

    def test_get_todos(self):
        todo_list = TodoList()
        todo = todo_list.add_todo('Buy milk')
        todos = todo_list.get_todos()
        self.assertEqual(len(todos), 1)


────────────────────────────────────────────────────────────
  tests/conftest.py


## 2. Health Check

In [4]:
services = {
    'test-runner': f'{TEST_RUNNER_URL}/health',
    'LiteLLM':     f'{LITELLM_URL}/health/liveliness',
    'n8n':         f'{N8N_URL}/healthz',
}
for name, url in services.items():
    try:
        r = requests.get(url, timeout=5)
        icon = '✅' if r.status_code == 200 else f'⚠️  {r.status_code}'
    except Exception as e:
        icon = f'❌  {e}'
    print(f'{icon}  {name}')

✅  test-runner
✅  LiteLLM
✅  n8n


## 3. Run Tests Directly
Runs pytest from inside this notebook (no LLM fix loop). Useful for a quick sanity check.

In [5]:
tests_dir = PROJECT_DIR_LOCAL / 'tests'

if not tests_dir.exists():
    print('No tests/ directory found in project.')
else:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_DIR_LOCAL)

    result = subprocess.run(
        ['python', '-m', 'pytest', 'tests/', '-v', '--tb=short', '--no-header'],
        capture_output=True, text=True,
        cwd=str(PROJECT_DIR_LOCAL), env=env,
    )
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    print(f'Exit code: {result.returncode}')


STDERR: /opt/conda/bin/python: No module named pytest

Exit code: 1


## 4. Trigger Phase 3 Loop
Calls the test-runner service directly (bypasses n8n for faster feedback).  
The loop runs up to 3 fix attempts if tests fail.

In [6]:
print('Calling test-runner /run ...  (may take several minutes if fixes are needed)\n')

try:
    r = requests.post(
        f'{TEST_RUNNER_URL}/run',
        json={'project_dir': PROJECT_DIR_RUNNER},
        timeout=600,
    )
    result = r.json()
except Exception as e:
    print(f'❌ Request failed: {e}')
    result = {}

# ── summary ──────────────────────────────────────────────────────────────────
passed     = result.get('passed', False)
iterations = result.get('iterations', '?')
icon       = '✅' if passed else '❌'

print(f'{icon}  Result     : {"PASSED" if passed else "FAILED"}')
print(f'   Iterations : {iterations}')
print()

# ── execution log ─────────────────────────────────────────────────────────────
for line in result.get('log', []):
    print(line)

Calling test-runner /run ...  (may take several minutes if fixes are needed)



✅  Result     : PASSED
   Iterations : 1

=== Phase 3 starting — /data/output/project ===
  venv: Deps unchanged — skipped install

── Test run 1  ──
  ✅ All tests passed!

Report written → /data/output/phase3_report.md


## 5. View Phase 3 Report

In [7]:
if REPORT_PATH.exists():
    from IPython.display import Markdown, display
    display(Markdown(REPORT_PATH.read_text()))
else:
    print('No report yet — run Phase 3 first (cell above).')

# Phase 3 Report — ✅ PASSED

**Project**: `/data/output/project`  
**Iterations**: 1  
**Result**: ✅ PASSED

## Attempt 1 — ✅ PASS
```
============================= test session starts ==============================
collecting ... collected 5 items

tests/test_open_meteo_weather.py::TestOpenMeteoWeather::test_get_weather_geocode_failure PASSED [ 20%]
tests/test_open_meteo_weather.py::TestOpenMeteoWeather::test_get_weather_success PASSED [ 40%]
tests/test_todo.py::TestTodo::test_todo_creation PASSED                  [ 60%]
tests/test_todo.py::TestTodoList::test_add_todo PASSED                   [ 80%]
tests/test_todo.py::TestTodoList::test_get_todos PASSED                  [100%]

============================== 5 passed in 0.07s ===============================

```


## 6. Inject a Bug
Deliberately break a source file to test the fix loop.  
Run the cell below, then re-trigger Phase 3 in section 7.

In [8]:
# Choose a target file — pick one that has tests covering it
TARGET = 'todo.py'
target_path = PROJECT_DIR_LOCAL / TARGET

if not target_path.exists():
    # Fall back to the first .py in the project root
    candidates = [f for f in PROJECT_DIR_LOCAL.glob('*.py')
                  if f.name not in ('conftest.py',)]
    if candidates:
        target_path = candidates[0]
        TARGET = target_path.name
    else:
        print('No suitable target file found.')
        target_path = None

if target_path:
    original = target_path.read_text()

    # Save original so we can restore it
    backup_path = target_path.with_suffix('.py.bak')
    backup_path.write_text(original)

    # Inject an off-by-one bug: make id_counter start at 0 instead of 1
    # (causes tests expecting id=1 to get id=0)
    bugged = original.replace('self.id_counter = 1', 'self.id_counter = 0')

    if bugged == original:
        # Generic fallback: append a syntax-breaking import
        bugged = 'import this_module_does_not_exist\n' + original

    target_path.write_text(bugged)
    print(f'✅ Bug injected into {TARGET}')
    print(f'   Backup saved to  {backup_path.name}')
    print()

    # Show what changed
    orig_lines   = original.splitlines()
    bugged_lines = bugged.splitlines()
    for i, (a, b) in enumerate(zip(orig_lines, bugged_lines)):
        if a != b:
            print(f'  Line {i+1} before: {a.strip()}')
            print(f'  Line {i+1} after : {b.strip()}')

✅ Bug injected into todo.py
   Backup saved to  todo.py.bak

  Line 17 before: self.id_counter = 1
  Line 17 after : self.id_counter = 0


## 7. Re-run Phase 3 — Watch the Fix Loop
The test-runner will detect the failure, call `free/code` for a fix, apply it, and re-run tests.

In [9]:
print('Running Phase 3 fix loop against the bugged project ...\n')

try:
    r = requests.post(
        f'{TEST_RUNNER_URL}/run',
        json={'project_dir': PROJECT_DIR_RUNNER},
        timeout=600,
    )
    result = r.json()
except Exception as e:
    print(f'❌ Request failed: {e}')
    result = {}

passed     = result.get('passed', False)
iterations = result.get('iterations', '?')
icon       = '✅' if passed else '❌'

print(f'{icon}  Result     : {"PASSED" if passed else "FAILED"}')
print(f'   Iterations : {iterations}  (fix loop used {max(0, int(str(iterations)) - 1)} LLM call(s))')
print()
for line in result.get('log', []):
    print(line)

Running Phase 3 fix loop against the bugged project ...



✅  Result     : PASSED
   Iterations : 1  (fix loop used 0 LLM call(s))

=== Phase 3 starting — /data/output/project ===
  venv: Deps unchanged — skipped install

── Test run 1  ──
  ✅ All tests passed!

Report written → /data/output/phase3_report.md


In [10]:
# Show the updated report
from IPython.display import Markdown, display
display(Markdown(REPORT_PATH.read_text()))

# Phase 3 Report — ✅ PASSED

**Project**: `/data/output/project`  
**Iterations**: 1  
**Result**: ✅ PASSED

## Attempt 1 — ✅ PASS
```
============================= test session starts ==============================
collecting ... collected 5 items

tests/test_open_meteo_weather.py::TestOpenMeteoWeather::test_get_weather_geocode_failure PASSED [ 20%]
tests/test_open_meteo_weather.py::TestOpenMeteoWeather::test_get_weather_success PASSED [ 40%]
tests/test_todo.py::TestTodo::test_todo_creation PASSED                  [ 60%]
tests/test_todo.py::TestTodoList::test_add_todo PASSED                   [ 80%]
tests/test_todo.py::TestTodoList::test_get_todos PASSED                  [100%]

============================== 5 passed in 0.07s ===============================

```


## 8. Restore Original Files
Restores any `.bak` backups created by the bug injector.

In [11]:
restored = []
for bak in PROJECT_DIR_LOCAL.glob('*.py.bak'):
    original_path = bak.with_suffix('')  # removes .bak → .py
    original_path.write_text(bak.read_text())
    bak.unlink()
    restored.append(original_path.name)

if restored:
    print(f'Restored: {restored}')
else:
    print('No backups to restore.')

Restored: ['todo.py']


## 9. Trigger via n8n Webhook
Use this cell to trigger Phase 3 through the n8n workflow (same as a real pipeline run).

In [12]:
r = requests.post(
    N8N_WEBHOOK,
    json={'project_dir': PROJECT_DIR_RUNNER},
    timeout=10,
)
print(f'Status : {r.status_code}')
print(f'Response: {r.json()}')
print()
print('Phase 3 is running asynchronously in n8n.')
print(f'Check the report at: {REPORT_PATH}')
print('Or view execution in the n8n UI: http://localhost:5678')

Status : 200
Response: {'message': 'Workflow was started'}

Phase 3 is running asynchronously in n8n.
Check the report at: /home/jovyan/output/phase3_report.md
Or view execution in the n8n UI: http://localhost:5678
